# Neural Topology Mapping — Crowd Surge & Compression Mitigation (Phase 2)
### Google Colab GPU Deployment with Cloudflare Tunnel

> **Important Notes on Google Colab Execution:**
- Free Google Colab instances allocate a dedicated NVIDIA Tesla T4 or A100 GPU.
- Keep this tab open while presenting to prevent Colab from entering an idle timeout.
- If you encounter any port conflict, re-running Step 3 will automatically terminate old processes and re-bind.

### Step 1: Verify Hardware Accelerator (NVIDIA CUDA GPU)

In [ ]:
!nvidia-smi

### Step 2: Clone / Update Repository and Install Dependencies

In [ ]:
# Clone repository or pull latest commits
!git clone https://github.com/rrohancodes/mini_project.git || (cd mini_project && git reset --hard && git pull)
%cd mini_project

# Install required dependencies including lapx solver for ByteTrack
!pip install -q -r requirements.txt ultralytics lapx


### Step 3: Launch FastAPI Server in Background (Port 8000)

In [ ]:
import subprocess
import time
import requests

# 1. Cleanly kill any previous background processes holding port 8000
!fuser -k 8000/tcp 2>/dev/null || pkill -f 'server.py' 2>/dev/null
time.sleep(1.5)

# 2. Launch FastAPI server.py as a background daemon
server_log = open('server.log', 'w')
server_proc = subprocess.Popen(
    ['python3', 'server.py', '--port', '8000'],
    stdout=server_log,
    stderr=subprocess.STDOUT
)

# 3. Poll server health check until online
print('Initializing Neural Topology server on port 8000...')
ready = False
for attempt in range(35):
    try:
        res = requests.get('http://localhost:8000/api/info', timeout=1)
        if res.status_code == 200:
            info = res.json()
            print('=' * 60)
            print('>>> SERVER STATUS : ONLINE <<<')
            print(f'>>> ACTIVE MODEL  : {info.get("model")}')
            print(f'>>> COMPUTE ENGINE: {info.get("device")}')
            print('=' * 60)
            ready = True
            break
    except Exception:
        time.sleep(1)

if not ready:
    print('Warning: Server did not respond within 35s. Diagnostic log:')
    !cat server.log


### Step 4: Expose Web Portal via Cloudflare Quick Tunnel (Public HTTPS)
Downloads the official Cloudflare tunnel binary and creates an encrypted zero-config HTTPS tunnel directly to your Colab runtime.

In [ ]:
import os
import re
import subprocess
import time

# 1. Download official cloudflared binary if not present
if not os.path.exists('cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

# 2. Launch tunnel pointing to localhost:8000
tunnel_log = open('tunnel.log', 'w')
tunnel_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT
)

# 3. Extract public HTTPS tunnel link
tunnel_url = None
for _ in range(25):
    time.sleep(1)
    with open('tunnel.log', 'r') as f:
        log_text = f.read()
    matches = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_text)
    if matches:
        tunnel_url = matches[0]
        break

if tunnel_url:
    print('=' * 65)
    print(f'  PUBLIC ACCESS PORTAL: {tunnel_url}')
    print('  (Click link above to open the CCTV dashboard in your browser)')
    print('=' * 65)
else:
    print('Could not find Cloudflare URL. Inspect tunnel.log below:')
    !cat tunnel.log


### Step 5: Live Diagnostic & Telemetry Log Stream

In [ ]:
# Display latest 40 lines of server and inference log
!tail -n 40 server.log
